# 1.0 EDA

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from project.config import RAW_DATA_DIR, PROCESSED_DATA_DIR

sns.set_theme(style="whitegrid")

# ---- コンペに合わせて変更 ----
TARGET_COL = "target"
ID_COL = "id"
# ----------------------------

train = pd.read_csv(RAW_DATA_DIR / "train.csv")
test  = pd.read_csv(RAW_DATA_DIR / "test.csv")

print(f"train: {train.shape}, test: {test.shape}")

## 基本情報

In [ ]:
train.head()

In [ ]:
train.info()

In [ ]:
train.describe()

## 欠損値

In [ ]:
missing = pd.DataFrame({
    "train": train.isnull().sum(),
    "test":  test.isnull().sum(),
})
missing[missing.sum(axis=1) > 0].sort_values("train", ascending=False)

## ターゲット分布

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

train[TARGET_COL].value_counts().plot.bar(ax=axes[0], rot=0)
axes[0].set_title("count")

train[TARGET_COL].value_counts(normalize=True).plot.pie(
    ax=axes[1], autopct="%1.1f%%", startangle=90
)
axes[1].set_ylabel("")
axes[1].set_title("ratio")

plt.tight_layout()
plt.show()

## 数値特徴量の分布 (train vs test)

In [ ]:
num_cols = train.select_dtypes(include="number").columns.drop([TARGET_COL, ID_COL], errors="ignore").tolist()

ncols = 3
nrows = -(-len(num_cols) // ncols)  # ceil
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    train[col].plot.hist(ax=ax, bins=40, alpha=0.6, label="train")
    test[col].plot.hist(ax=ax, bins=40, alpha=0.6, label="test")
    ax.set_title(col)
    ax.legend()

for ax in axes[len(num_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## 数値特徴量 × ターゲット

In [ ]:
fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4 * nrows))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    for t, grp in train.groupby(TARGET_COL):
        grp[col].plot.hist(ax=ax, bins=40, alpha=0.5, label=f"{TARGET_COL}={t}")
    ax.set_title(col)
    ax.legend()

for ax in axes[len(num_cols):]:
    ax.set_visible(False)

plt.tight_layout()
plt.show()

## カテゴリ特徴量 × ターゲット

In [ ]:
cat_cols = train.select_dtypes(include="object").columns.tolist()

if cat_cols:
    ncols_c = min(3, len(cat_cols))
    nrows_c = -(-len(cat_cols) // ncols_c)
    fig, axes = plt.subplots(nrows_c, ncols_c, figsize=(5 * ncols_c, 4 * nrows_c))
    axes = axes.flatten() if len(cat_cols) > 1 else [axes]

    for ax, col in zip(axes, cat_cols):
        ct = pd.crosstab(train[col], train[TARGET_COL], normalize="index")
        ct.plot.bar(ax=ax, stacked=True, rot=45)
        ax.set_title(f"{col}")
        ax.legend(title=TARGET_COL)

    for ax in axes[len(cat_cols):]:
        ax.set_visible(False)

    plt.tight_layout()
    plt.show()
else:
    print("カテゴリ列なし")

## 相関行列

In [ ]:
corr = train.select_dtypes(include="number").drop(ID_COL, axis=1, errors="ignore").corr()

plt.figure(figsize=(max(8, len(corr)), max(6, len(corr) - 1)))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()